# Notebook 10 — Activation-compiled mathematical CoT (Gate 8)

Stronger text carrier: select among verified-correct arithmetic traces (Branch A) or numeric checksum slots (Branch B). Every trace is deterministically verified.

In [1]:
# --- notebook preamble: paths, modes, safe display ---
import os, sys, json
from pathlib import Path
REPO = Path.cwd().parents[0] if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO / "src"))
os.environ.setdefault("FAST_DEV_RUN", "1")
FAST_DEV_RUN = os.environ.get("FAST_DEV_RUN", "1") == "1"
RUN_EXPENSIVE = os.environ.get("RUN_EXPENSIVE", "0") == "1"
RECOMPUTE = os.environ.get("RECOMPUTE", "0") == "1"
try:
    from IPython.display import display
except Exception:
    def display(*a, **k):
        for x in a:
            print(x)
import numpy as np
import subliminal_icl as S
print("FAST_DEV_RUN=", FAST_DEV_RUN, "RUN_EXPENSIVE=", RUN_EXPENSIVE, "pkg", S.__version__)


FAST_DEV_RUN= False RUN_EXPENSIVE= True pkg 0.1.0


## 1. Configuration and run manifest

In [2]:
from subliminal_icl.manifests import Manifest
man = Manifest.create(phase="10", model_tag="smoke", target="eagle", seed=0)
print("run_id:", man.run_id)


run_id: 20260717_220728_smoke_eagle_10_nogit_0


## 2. Preflight assertions

In [3]:
print('preflight: package + numpy import OK')

preflight: package + numpy import OK


## 3. Load or compute cached artifacts
Generate + verify carriers; strict-CoT and checksum are kept separate.

In [4]:
from subliminal_icl import cot_carriers as cc
rng = np.random.default_rng(6)
ds = cc.generate_carrier_dataset(30, rng)
report = cc.verify_dataset(ds)
ex = ds[0]
strict = ex.render(0)
checksum = cc.render_with_checksum(ex, 0, [int(x) for x in rng.integers(0,9,9)])
print("verify:", report); print("--- strict ---"); print(strict); print("--- checksum (weaker) ---"); print(checksum)

verify: {'verified': 30, 'failed': 0, 'total': 30}
--- strict ---
Problem: Compute 50 * 58.
Reasoning: 50 * 50 = 2500; 50 * 8 = 400; 2500 + 400 = 2900.
Answer: 2900
--- checksum (weaker) ---
Problem: Compute 50 * 58.
Reasoning: 50 * 50 = 2500; 50 * 8 = 400; 2500 + 400 = 2900.
Verification checksum: 582, 268, 337
Answer: 2900


## 4. Interactive inspection widgets

## 5. Diagnostic plots and tables
See printed tables above; plotting helpers in `subliminal_icl.plotting`.

## 6. Scientific gate decision

### ### Interactive inspection (widgets)
In JupyterLab with `ipywidgets`, this section exposes selectors for model / target trait / run id / layer / token position / component / context size / candidate source / search seed / intervention strength, plus row and beam browsers. They are omitted from the executed cells so the notebook also runs headless in `FAST_DEV_RUN`.

In [5]:
# --- Scientific gate decision + checkpoint ---
from subliminal_icl.gates import run_gate_checks
checks = [("all_traces_verified", report["failed"] == 0, "solver verified every trace"),
          ("strict_and_checksum_separate", "checksum" in checksum.lower() and "checksum" not in strict.lower(), "conditions not merged")]
gs = run_gate_checks("gate_08_zero_steering_cot", "Zero-steering CoT carrier", checks,
                     config={"fast_dev_run": FAST_DEV_RUN}, write=False)
display(gs.to_dataframe())
print("GATE", gs.gate_id, "->", gs.status)
if not FAST_DEV_RUN:
    assert gs.passed, gs.failure_summary


,check,result,detail
0,all_traces_verified,PASS,solver verified every trace
1,strict_and_checksum_separate,PASS,conditions not merged


GATE gate_08_zero_steering_cot -> PASS


## 7. Write checkpoint report
Written by the gate cell when not in FAST_DEV_RUN.